In [ ]:
# 02_results_viewer.ipynb
# PURPOSE
#   View all evaluation results stored in the checkpoint.
#   No model loading. No inference. No GPU needed.
#   Run this any time you want to see current results.
#
# RUNTIME
#   ~30 seconds. Reads from Drive only.
#
# HOW TO USE
#   Run all cells top to bottom.
#   Results appear for every model that has been evaluated so far.
#   As you add new models in 01_train_and_evaluate.ipynb,
#   they automatically appear here next time you run this.

In [ ]:
#Cell 1: Load everything from checkpoint

import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR     = Path("/content/drive/MyDrive/LLM_to_API_Project/")
CHECKPOINT_PATH = PROJECT_DIR / "eval_results" / "hybrid_checkpoint.pkl"
CONFIDENCE_THRESHOLD = 0.6

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "No checkpoint found. Run 01_train_and_evaluate.ipynb first."
    )

with open(CHECKPOINT_PATH, "rb") as f:
    ck = pickle.load(f)

y_true     = ck["y_eval_labels"]
conf_arr   = np.array(ck["hybrid_confidences"])
high_idx   = np.where(conf_arr >= CONFIDENCE_THRESHOLD)[0]
low_idx    = np.where(conf_arr <  CONFIDENCE_THRESHOLD)[0]

print(f"Checkpoint loaded ✅")
print(f"Eval set       : {len(y_true)} reports")
print(f"High confidence: {len(high_idx)} ({len(high_idx)/len(y_true)*100:.1f}%)")
print(f"Low confidence : {len(low_idx)} ({len(low_idx)/len(y_true)*100:.1f}%)")
print(f"\nModels in checkpoint:")

# Registry — add new model keys here as they are evaluated
MODEL_REGISTRY = {
    "Hybrid"       : ck.get("hybrid_preds"),
    "Qwen2.5-1.5B" : ck.get("qwen15b_preds"),
    "Qwen2.5-3B"   : ck.get("qwen3b_preds"),
    "Qwen2.5-7B ★" : ck.get("qwen7b_preds"),
    "Claude Haiku" : ck.get("haiku_preds"),
}
available = {k: v for k, v in MODEL_REGISTRY.items() if v is not None}

for name in MODEL_REGISTRY:
    status = "✅ evaluated" if MODEL_REGISTRY[name] else "⬜ not yet run"
    print(f"  {name:<22} {status}")

if ck.get("groq_sub_preds"):
    print(f"  {'Groq Llama3.1-8B':<22} ✅ evaluated (N=300 subset)")

In [ ]:
# Cell 2: Accuracy comparison table
print("\"ACCURACY COMPARISON — ALL EVALUATED MODELS\"")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")

for group_name, indices in [
    ("ALL CASES",       np.arange(len(y_true))),
    ("HIGH CONFIDENCE", high_idx),
    ("LOW CONFIDENCE",  low_idx),
]:
    if len(indices) == 0:
        continue
    gt    = [y_true[i] for i in indices]
    h_acc = accuracy_score(gt, [available["Hybrid"][i] for i in indices])

    print(f"\n{group_name}  "
          f"(N={len(indices)}, {len(indices)/len(y_true)*100:.1f}%)")
    print(f"  {'Model':<22}  {'Accuracy':>10}  {'Macro-F1':>10}  {'vs Hybrid':>10}")

    for name, preds in available.items():
        p     = [preds[i] for i in indices]
        acc   = accuracy_score(gt, p)
        mf1   = f1_score(gt, p, average="macro", zero_division=0)
        delta = acc - h_acc
        ds    = "    —    " if name == "Hybrid" else f"{delta:+.4f}"
        print(f"  {name:<22}  {acc:>10.4f}  {mf1:>10.4f}  {ds:>10}")

# Groq subset (separate because N=300)
if ck.get("groq_sub_preds") and ck.get("groq_sub_idx"):
    sub_idx  = ck["groq_sub_idx"]
    conf_sub = np.array([conf_arr[i] for i in sub_idx])
    low_sub  = np.where(conf_sub < CONFIDENCE_THRESHOLD)[0]
    y_sub    = [y_true[i] for i in sub_idx]
    h_sub    = [ck["hybrid_preds"][i] for i in sub_idx]
    g_sub    = ck["groq_sub_preds"]

    print(f"\n\nGROQ LLAMA 3.1 8B — N=300 SUBSET")
    print("Note: full evaluation not possible due to 500K tokens/day limit")
    for group_name, indices in [
        ("ALL CASES",      np.arange(300)),
        ("LOW CONFIDENCE", low_sub),
    ]:
        gt    = [y_sub[i] for i in indices]
        h_acc = accuracy_score(gt, [h_sub[i] for i in indices])
        g_acc = accuracy_score(gt, [g_sub[i] for i in indices])
        g_mf1 = f1_score(gt, [g_sub[i] for i in indices],
                         average="macro", zero_division=0)
        print(f"\n{group_name} (N={len(indices)})")
        print(f"  {'Hybrid':<22}  {h_acc:>10.4f}")
        print(f"  {'Groq Llama3.1-8B':<22}  {g_acc:>10.4f}  "
              f"{g_mf1:>10.4f}  {g_acc-h_acc:>+10.4f}")

In [ ]:
# Cell 3: Recovery and regression analysis
print("RECOVERY ANALYSIS — LOW-CONFIDENCE CASES")
print("How many hybrid errors does each fallback model recover?")

hybrid_preds = ck["hybrid_preds"]

for name, preds in available.items():
    if name == "Hybrid":
        continue
    rec = sum(1 for i in low_idx
              if hybrid_preds[i] != y_true[i] and preds[i] == y_true[i])
    reg = sum(1 for i in low_idx
              if hybrid_preds[i] == y_true[i] and preds[i] != y_true[i])
    both = sum(1 for i in low_idx
               if hybrid_preds[i] != y_true[i] and preds[i] != y_true[i])
    print(f"\n{name}")
    print(f"  Recoveries  (hybrid wrong → model right): {rec}")
    print(f"  Regressions (hybrid right → model wrong): {reg}")
    print(f"  Both wrong                              : {both}")
    print(f"  Net gain                                : {rec-reg:+d}")

if ck.get("groq_sub_preds"):
    sub_idx = ck["groq_sub_idx"]
    y_sub   = [y_true[i] for i in sub_idx]
    h_sub   = [hybrid_preds[i] for i in sub_idx]
    g_sub   = ck["groq_sub_preds"]
    conf_sub = np.array([conf_arr[i] for i in sub_idx])
    low_sub  = np.where(conf_sub < CONFIDENCE_THRESHOLD)[0]

    rec  = sum(1 for i in low_sub if h_sub[i] != y_sub[i] and g_sub[i] == y_sub[i])
    reg  = sum(1 for i in low_sub if h_sub[i] == y_sub[i] and g_sub[i] != y_sub[i])
    both = sum(1 for i in low_sub if h_sub[i] != y_sub[i] and g_sub[i] != y_sub[i])
    print(f"\nGroq Llama3.1-8B (N=300 subset)")
    print(f"  Recoveries  : {rec}")
    print(f"  Regressions : {reg}")
    print(f"  Both wrong  : {both}")
    print(f"  Net gain    : {rec-reg:+d}")

In [ ]:
#Cell 4: Decision summary
print("DECISION SUMMARY!")

# Find the best local model (highest low-conf accuracy among non-API models)
local_models = {
    k: v for k, v in available.items()
    if k not in ("Hybrid", "Claude Haiku")
}

best_name, best_acc, best_net = None, 0, 0
for name, preds in local_models.items():
    gt  = [y_true[i] for i in low_idx]
    acc = accuracy_score(gt, [preds[i] for i in low_idx])
    rec = sum(1 for i in low_idx
              if hybrid_preds[i] != y_true[i] and preds[i] == y_true[i])
    reg = sum(1 for i in low_idx
              if hybrid_preds[i] == y_true[i] and preds[i] != y_true[i])
    net = rec - reg
    if acc > best_acc:
        best_acc, best_name, best_net = acc, name, net

print(f"\nChosen fallback : {best_name}")
print(f"  Low-confidence accuracy : {best_acc:.1%}  (N={len(low_idx)})")
print(f"  Net gain over hybrid    : {best_net:+d} cases")

# All models ranked by low-confidence accuracy
print(f"\nAll models ranked by low-confidence accuracy:")
print(f"  {'Model':<22}  {'Low-conf acc':>14}  {'Net gain':>10}")
for name, preds in available.items():
    gt  = [y_true[i] for i in low_idx]
    acc = accuracy_score(gt, [preds[i] for i in low_idx])
    rec = sum(1 for i in low_idx
              if hybrid_preds[i] != y_true[i] and preds[i] == y_true[i])
    reg = sum(1 for i in low_idx
              if hybrid_preds[i] == y_true[i] and preds[i] != y_true[i])
    net = rec - reg if name != "Hybrid" else 0
    marker = " ★" if name == best_name else ""
    print(f"  {name+marker:<22}  {acc:>14.1%}  {net:>+10d}")

if ck.get("groq_sub_preds"):
    sub_idx  = ck["groq_sub_idx"]
    y_sub    = [y_true[i] for i in sub_idx]
    h_sub    = [hybrid_preds[i] for i in sub_idx]
    g_sub    = ck["groq_sub_preds"]
    conf_sub = np.array([conf_arr[i] for i in sub_idx])
    low_sub  = np.where(conf_sub < CONFIDENCE_THRESHOLD)[0]
    gt       = [y_sub[i] for i in low_sub]
    g_acc    = accuracy_score(gt, [g_sub[i] for i in low_sub])
    rec      = sum(1 for i in low_sub
                   if h_sub[i] != y_sub[i] and g_sub[i] == y_sub[i])
    reg      = sum(1 for i in low_sub
                   if h_sub[i] == y_sub[i] and g_sub[i] != y_sub[i])
    print(f"  {'Groq Llama3.1-8B*':<22}  {g_acc:>14.1%}  {rec-reg:>+10d}")
    print(f"  * N=300 subset — full eval limited by 500K tokens/day free tier")